<a href="https://colab.research.google.com/github/Abhi712-ui/lis501-ipynb-files/blob/main/LIS501_Week_4_BagofWords%2C_TF_IDF%2C_Text_Similarity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 · One corpus, three ways to represent and compare text
## Bag-of-Words → TF-IDF → Text similarity

Run the cells from top to bottom in Google Colab. The corpus is embedded below; no dataset upload or pretrained language model is needed. The main path uses the same rules and numbers as the slides. A separately labeled optional section shows the scikit-learn default TF-IDF convention.

**Our question:** Which descriptions share vocabulary about similar uses of campus spaces? Read the descriptions first and predict some close pairs.

**Fixed classroom convention:** lowercase; extract `[a-z]+` tokens; remove only `the`, `is`, `has`, and `and`; keep surface forms and repetitions. Use relative TF, `log10(N / df)` IDF, and no additional normalization of the displayed weights. Cosine similarity normalizes vector lengths when comparing them.

In [ ]:
import re
import sys
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
# Colab normally provides these packages. If an import fails, install
# the missing package in a separate cell, then rerun this cell.

Python: 3.13.15
scikit-learn: 1.6.1


## 1. Keep the same six documents throughout
**Lecture slides 3–4.** A document can contain one sentence or several sentences. D1 contains two sentences but still counts as one document. The labels below identify texts; they are not model predictions.

In [ ]:
documents = {
    "D1": "The campus library is quiet. The library has quiet study space.",
    "D2": "The campus library has study space.",
    "D3": "The campus cafe has coffee and study space.",
    "D4": "The campus cafe has coffee and food.",
    "D5": "The campus gym has fitness classes.",
    "D6": "The campus gym has fitness classes and equipment."
}

doc_ids = list(documents)
texts = list(documents.values())
display(pd.DataFrame({"document": doc_ids, "original_text": texts}))

,document,original_text
0,D1,The campus library is quiet. The library has q...
1,D2,The campus library has study space.
2,D3,The campus cafe has coffee and study space.
3,D4,The campus cafe has coffee and food.
4,D5,The campus gym has fitness classes.
5,D6,The campus gym has fitness classes and equipment.


### Make the preprocessing rule visible
We reuse Regex from Week 3 to extract words. The four-word stopword list is a deliberate rule for this small English example. It is not a general-purpose preprocessing recipe. `campus` stays in the texts so we can observe what corpus-wide frequency does. We do not stem or lemmatize; `classes` stays `classes`.

In [ ]:
stop_words = {"the", "is", "has", "and"}

def preprocess(text):
    """Return selected surface-form tokens for this English classroom corpus."""
    if not isinstance(text, str):
        raise TypeError("preprocess expects a string.")
    words = re.findall(r"[a-z]+", text.lower())
    kept_words = []
    for word in words:
        if word not in stop_words:
            kept_words.append(word)
    return kept_words

token_lists = []
cleaned_texts = []
for text in texts:
    tokens = preprocess(text)
    token_lists.append(tokens)
    cleaned_texts.append(" ".join(tokens))

display(pd.DataFrame({
    "document": doc_ids,
    "selected_terms": cleaned_texts,
    "retained_tokens": [len(tokens) for tokens in token_lists]
}))

,document,selected_terms,retained_tokens
0,D1,campus library quiet library quiet study space,7
1,D2,campus library study space,4
2,D3,campus cafe coffee study space,5
3,D4,campus cafe coffee food,4
4,D5,campus gym fitness classes,4
5,D6,campus gym fitness classes equipment,5


## 2. Bag-of-Words: make the vocabulary, then count
**Lecture slides 5–7.** Build the vocabulary in **first-appearance order across D1–D6**. Every document will use these same 12 columns. A vocabulary contains each term once; each document's token list preserves repetitions.

In [ ]:
vocabulary = []
for tokens in token_lists:
    for word in tokens:
        if word not in vocabulary:
            vocabulary.append(word)

print("Vocabulary:", vocabulary)
print("Number of features:", len(vocabulary))

Vocabulary: ['campus', 'library', 'quiet', 'study', 'space', 'cafe', 'coffee', 'food', 'gym', 'fitness', 'classes', 'equipment']
Number of features: 12


In [ ]:
rows = []
for tokens in token_lists:
    row = []
    for word in vocabulary:
        row.append(tokens.count(word))
    rows.append(row)

counts = np.array(rows)
count_table = pd.DataFrame(counts, index=doc_ids, columns=vocabulary)
display(count_table)
print("Shape (documents, features):", counts.shape)

,campus,library,quiet,study,space,cafe,coffee,food,gym,fitness,classes,equipment
D1,1,2,2,1,1,0,0,0,0,0,0,0
D2,1,1,0,1,1,0,0,0,0,0,0,0
D3,1,0,0,1,1,1,1,0,0,0,0,0
D4,1,0,0,0,0,1,1,1,0,0,0,0
D5,1,0,0,0,0,0,0,0,1,1,1,0
D6,1,0,0,0,0,0,0,0,1,1,1,1


Shape (documents, features): (6, 12)


**Predict:** D1 has seven retained tokens and five distinct terms. Which entries record the repeated words? What would change if one extra `library` token were added to D2?

### Reproduce that exact table with CountVectorizer
The text is already processed, so we split at spaces and explicitly reuse our vocabulary. This prevents a second, different tokenization or column ordering from changing the example. The library returns a sparse matrix; converting to a dense array is convenient for this tiny corpus, not a general rule for large collections. [CountVectorizer documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html).

In [ ]:
vectorizer = CountVectorizer(
    tokenizer=str.split,
    token_pattern=None,
    lowercase=False,
    vocabulary=vocabulary
)
X_counts = vectorizer.fit_transform(cleaned_texts)
np.testing.assert_array_equal(X_counts.toarray(), counts)
print("The hand-built and library count matrices match exactly.")

The hand-built and library count matrices match exactly.


## 3. TF-IDF: keep the columns and change the weights
**Lecture slides 8–13.** We retain Week 3's calculation:

\[
TF(t,d)=\frac{\operatorname{count}(t,d)}{\text{retained tokens in }d},\qquad
IDF(t)=\log_{10}\left(\frac{N}{df(t)}\right),\qquad
w(t,d)=TF(t,d)\times IDF(t).
\]

The denominator for D1 is **7**, not the number of words in its original sentences. There are **6 documents** in the corpus.

### Term frequency: repetitions within a document
`axis=1` means sum across the columns in each row. `[:, None]` places the row totals in a column so NumPy can divide each row by its own total.

In [ ]:
document_lengths = counts.sum(axis=1)
if np.any(document_lengths == 0):
    raise ValueError("At least one document has no retained tokens.")
term_frequency = counts / document_lengths[:, None]
display(pd.DataFrame(term_frequency, index=doc_ids, columns=vocabulary).round(3))

,campus,library,quiet,study,space,cafe,coffee,food,gym,fitness,classes,equipment
D1,0.143,0.286,0.286,0.143,0.143,0.00,0.00,0.00,0.00,0.00,0.00,0.0
D2,0.250,0.250,0.000,0.250,0.250,0.00,0.00,0.00,0.00,0.00,0.00,0.0
D3,0.200,0.000,0.000,0.200,0.200,0.20,0.20,0.00,0.00,0.00,0.00,0.0
D4,0.250,0.000,0.000,0.000,0.000,0.25,0.25,0.25,0.00,0.00,0.00,0.0
D5,0.250,0.000,0.000,0.000,0.000,0.00,0.00,0.00,0.25,0.25,0.25,0.0
D6,0.200,0.000,0.000,0.000,0.000,0.00,0.00,0.00,0.20,0.20,0.20,0.2


### Document frequency: how many documents contain the term?
`counts > 0` converts counts into presence/absence. Summing down a column (`axis=0`) then counts documents. Two mentions of `quiet` in D1 contribute **one** to its document frequency.

In [ ]:
present = counts > 0
document_frequency = present.sum(axis=0)
corpus_occurrences = counts.sum(axis=0)
N = len(doc_ids)

frequency_table = pd.DataFrame({
    "corpus_occurrences": corpus_occurrences,
    "documents_containing_term": document_frequency
}, index=vocabulary)
display(frequency_table)

,corpus_occurrences,documents_containing_term
campus,6,6
library,3,2
quiet,2,1
study,3,3
space,3,3
cafe,2,2
coffee,2,2
food,1,1
gym,2,2
fitness,2,2


### Inverse document frequency
All vocabulary terms occur at least once in this fixed corpus. Under our classroom formula, `campus` has IDF zero because it occurs in every document. `quiet` has a higher IDF than `library` because it occurs in fewer documents.

In [ ]:
if np.any(document_frequency == 0):
    raise ValueError("Each vocabulary term must occur in the fitted corpus.")
inverse_document_frequency = np.log10(N / document_frequency)
display(pd.DataFrame({
    "df": document_frequency,
    "idf_log10": inverse_document_frequency
}, index=vocabulary).round(3))

,df,idf_log10
campus,6,0.000
library,2,0.477
quiet,1,0.778
study,3,0.301
space,3,0.301
cafe,2,0.477
coffee,2,0.477
food,1,0.778
gym,2,0.477
fitness,2,0.477


### Multiply each TF column by that term's IDF
Keep the full-precision values for calculations; round only the displayed tables.

In [ ]:
tfidf_classroom = term_frequency * inverse_document_frequency
tfidf_table = pd.DataFrame(tfidf_classroom, index=doc_ids, columns=vocabulary)
display(tfidf_table.round(3))

d1_details = pd.DataFrame({
    "count": counts[0],
    "tf": term_frequency[0],
    "df": document_frequency,
    "idf": inverse_document_frequency,
    "tfidf": tfidf_classroom[0]
}, index=vocabulary)
display(d1_details.loc[d1_details["count"] > 0].round(3))

,campus,library,quiet,study,space,cafe,coffee,food,gym,fitness,classes,equipment
D1,0.0,0.136,0.222,0.043,0.043,0.000,0.000,0.000,0.000,0.000,0.000,0.000
D2,0.0,0.119,0.000,0.075,0.075,0.000,0.000,0.000,0.000,0.000,0.000,0.000
D3,0.0,0.000,0.000,0.060,0.060,0.095,0.095,0.000,0.000,0.000,0.000,0.000
D4,0.0,0.000,0.000,0.000,0.000,0.119,0.119,0.195,0.000,0.000,0.000,0.000
D5,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.119,0.119,0.119,0.000
D6,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.095,0.095,0.095,0.156


,count,tf,df,idf,tfidf
campus,1,0.143,6,0.000,0.000
library,2,0.286,2,0.477,0.136
quiet,2,0.286,1,0.778,0.222
study,1,0.143,3,0.301,0.043
space,1,0.143,3,0.301,0.043


**Interpret:** `library` and `quiet` have identical counts in D1, but different TF-IDF weights. `campus` still describes the corpus even though its classroom weight is zero. The weight concerns how a word distinguishes documents in this chosen collection, not whether the word is meaningful.

## 4. Cosine similarity: compare vectors
**Lecture slides 14–20.** Cosine similarity divides a dot product by the product of vector lengths. These nonnegative document vectors produce scores between 0 and 1. These scores are not percentages of shared meaning. [Cosine documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html).

### First compare raw count vectors

In [ ]:
cosine_counts = cosine_similarity(counts)
display(pd.DataFrame(cosine_counts, index=doc_ids, columns=doc_ids).round(3))

a = counts[0]  # D1
b = counts[1]  # D2
dot_product = np.dot(a, b)
length_a = np.linalg.norm(a)
length_b = np.linalg.norm(b)
hand_cosine = dot_product / (length_a * length_b)
print("D1 · D2 =", dot_product)
print("Vector lengths:", length_a, length_b)
print("Cosine(D1, D2) =", round(hand_cosine, 3))

,D1,D2,D3,D4,D5,D6
D1,1.000,0.754,0.405,0.151,0.151,0.135
D2,0.754,1.000,0.671,0.250,0.250,0.224
D3,0.405,0.671,1.000,0.671,0.224,0.200
D4,0.151,0.250,0.671,1.000,0.250,0.224
D5,0.151,0.250,0.224,0.250,1.000,0.894
D6,0.135,0.224,0.200,0.224,0.894,1.000


D1 · D2 = 5
Vector lengths: 3.3166247903554 2.0
Cosine(D1, D2) = 0.754


### What happens when every count in D2 doubles?
This is a hypothetical scaled vector. We do not add a seventh document or refit the corpus. Compare this with changing the count of just one term later.

In [ ]:
doubled_d2 = 2 * counts[1]
original_score = cosine_similarity([counts[0]], [counts[1]])[0, 0]
doubled_score = cosine_similarity([counts[0]], [doubled_d2])[0, 0]
print("D1 versus D2:", round(original_score, 3))
print("D1 versus 2 × D2:", round(doubled_score, 3))
np.testing.assert_allclose(original_score, doubled_score)

D1 versus D2: 0.754
D1 versus 2 × D2: 0.754


### Now compare TF-IDF vectors using the same cosine rule
Every number below uses all 12 features. A zero in this table means no overlap in positive-weight features, not no possible semantic connection.

In [ ]:
cosine_tfidf = cosine_similarity(tfidf_classroom)
display(pd.DataFrame(cosine_tfidf, index=doc_ids, columns=doc_ids).round(3))

comparison_rows = []
for left, right in [("D2", "D3"), ("D3", "D4"), ("D2", "D5")]:
    i = doc_ids.index(left)
    j = doc_ids.index(right)
    comparison_rows.append({
        "pair": left + "–" + right,
        "cosine_counts": cosine_counts[i, j],
        "cosine_classroom_tfidf": cosine_tfidf[i, j]
    })
display(pd.DataFrame(comparison_rows).round(3))

,D1,D2,D3,D4,D5,D6
D1,1.000,0.531,0.121,0.000,0.000,0.000
D2,0.531,1.000,0.355,0.000,0.000,0.000
D3,0.121,0.355,1.000,0.554,0.000,0.000
D4,0.000,0.000,0.554,1.000,0.000,0.000
D5,0.000,0.000,0.000,0.000,1.000,0.728
D6,0.000,0.000,0.000,0.000,0.728,1.000


,pair,cosine_counts,cosine_classroom_tfidf
0,D2–D3,0.671,0.355
1,D3–D4,0.671,0.554
2,D2–D5,0.250,0.000


**Notice the changed relationship:** D3 ties with D2 and D4 under count cosine (0.671 each). Under our TF-IDF convention, D3 is closer to D4 (0.554) than D2 (0.355). Shared `cafe`/`coffee` receive larger IDF weights than `study`/`space`. The shared `campus` term has zero weight. This is a result of this corpus and representation, not a guarantee that TF-IDF always improves a ranking.

## 5. Jaccard similarity: compare sets of retained words
**Lecture slides 21–23.** Use the original retained-token sets, before TF-IDF weighting. This means `campus` remains in each set. A set keeps distinct words and discards repetition.

\[
J(A,B)=\frac{|A\cap B|}{|A\cup B|}.
\]

The function raises an error for two empty sets so that a missing representation is not silently presented as meaningful similarity.

In [ ]:
def jaccard_similarity(tokens_a, tokens_b):
    """Compute ordinary, unweighted Jaccard on sets of retained tokens."""
    a = set(tokens_a)
    b = set(tokens_b)
    union = a | b
    if not union:
        raise ValueError("Jaccard is undefined here for two empty sets.")
    return len(a & b) / len(union)

A = set(token_lists[0])
B = set(token_lists[1])
print("D1 terms:", sorted(A))
print("D2 terms:", sorted(B))
print("Intersection:", sorted(A & B))
print("Union:", sorted(A | B))
print("Jaccard(D1, D2):", jaccard_similarity(A, B))

D1 terms: ['campus', 'library', 'quiet', 'space', 'study']
D2 terms: ['campus', 'library', 'space', 'study']
Intersection: ['campus', 'library', 'space', 'study']
Union: ['campus', 'library', 'quiet', 'space', 'study']
Jaccard(D1, D2): 0.8


In [ ]:
jaccard_matrix = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        jaccard_matrix[i, j] = jaccard_similarity(token_lists[i], token_lists[j])
display(pd.DataFrame(jaccard_matrix, index=doc_ids, columns=doc_ids).round(3))

,D1,D2,D3,D4,D5,D6
D1,1.000,0.800,0.429,0.125,0.125,0.111
D2,0.800,1.000,0.500,0.143,0.143,0.125
D3,0.429,0.500,1.000,0.500,0.125,0.111
D4,0.125,0.143,0.500,1.000,0.143,0.125
D5,0.125,0.143,0.125,0.143,1.000,0.800
D6,0.111,0.125,0.111,0.125,0.800,1.000


### Repeat only `library` in D2
This edited vector is for an isolated comparison; the original six-document corpus and fitted IDF remain unchanged. Predict which of these two scores changes.

In [ ]:
edited_d2_tokens = token_lists[1] + ["library"]
edited_d2_vector = vectorizer.transform([" ".join(edited_d2_tokens)]).toarray()[0]

display(pd.DataFrame({
    "comparison": ["D1 vs original D2", "D1 vs D2 + one library"],
    "count_cosine": [
        cosine_counts[0, 1],
        cosine_similarity([counts[0]], [edited_d2_vector])[0, 0]
    ],
    "jaccard": [
        jaccard_similarity(token_lists[0], token_lists[1]),
        jaccard_similarity(token_lists[0], edited_d2_tokens)
    ]
}).round(3))

,comparison,count_cosine,jaccard
0,D1 vs original D2,0.754,0.8
1,D1 vs D2 + one library,0.798,0.8


## 6. Use the same representation for a small search example
**Lecture slide 24.** Query: `quiet study space`.

The query is transformed with the **existing vocabulary and six-document IDF**. It is not a seventh fitted document. Unknown words are reported and omitted from this fixed feature space. An all-zero weighted query has no cosine direction, so the helper stops and explains the issue.

In [ ]:
def rank_documents(query):
    query_tokens = preprocess(query)
    unknown = sorted(set(query_tokens) - set(vocabulary))
    if unknown:
        print("Outside the fitted vocabulary:", unknown)
    query_counts = vectorizer.transform([" ".join(query_tokens)]).toarray()
    total = query_counts.sum()
    if total == 0:
        print("No query terms match the fitted vocabulary.")
        return pd.DataFrame(columns=["document", "score", "text"])
    query_tf = query_counts / total
    query_weights = query_tf * inverse_document_frequency
    if np.linalg.norm(query_weights) == 0:
        print("The query has no positive-weight terms under the classroom IDF.")
        return pd.DataFrame(columns=["document", "score", "text"])
    scores = cosine_similarity(query_weights, tfidf_classroom)[0]
    results = pd.DataFrame({"document": doc_ids, "score": scores, "text": texts})
    return results.sort_values("score", ascending=False, kind="stable").reset_index(drop=True)

search_results = rank_documents("quiet study space")
display(search_results.round(3))

,document,score,text
0,D1,0.837,The campus library is quiet. The library has q...
1,D2,0.320,The campus library has study space.
2,D3,0.256,The campus cafe has coffee and study space.
3,D4,0.000,The campus cafe has coffee and food.
4,D5,0.000,The campus gym has fitness classes.
5,D6,0.000,The campus gym has fitness classes and equipment.


## 7. Check the shared example
This cell verifies the lecture's matrices, formulas, comparison scores, and search scores against full-precision calculations. Assertions are instructor checks; students can focus on the displayed examples.

In [ ]:
expected_counts = np.array([[1, 2, 2, 1, 1, 0, 0, 0, 0, 0, 0, 0], [1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0], [1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1]])
np.testing.assert_array_equal(counts, expected_counts)
np.testing.assert_array_equal(document_lengths, [7, 4, 5, 4, 4, 5])
np.testing.assert_allclose(term_frequency.sum(axis=1), 1)
assert vocabulary == ['campus', 'library', 'quiet', 'study', 'space', 'cafe', 'coffee', 'food', 'gym', 'fitness', 'classes', 'equipment']
assert document_frequency[vocabulary.index("campus")] == 6
assert document_frequency[vocabulary.index("library")] == 2
assert document_frequency[vocabulary.index("quiet")] == 1
np.testing.assert_allclose(tfidf_classroom[:, vocabulary.index("campus")], 0)
np.testing.assert_allclose(cosine_counts[0, 1], 5 / np.sqrt(44))
np.testing.assert_allclose(cosine_counts[1, 2], 3 / np.sqrt(20))
np.testing.assert_allclose(cosine_counts[2, 3], 3 / np.sqrt(20))
assert round(cosine_tfidf[1, 2], 3) == 0.355
assert round(cosine_tfidf[2, 3], 3) == 0.554
assert jaccard_matrix[0, 1] == 0.8
np.testing.assert_allclose(cosine_counts, cosine_counts.T)
np.testing.assert_allclose(cosine_tfidf, cosine_tfidf.T)
np.testing.assert_allclose(np.diag(cosine_tfidf), 1)
assert search_results["document"].tolist()[:3] == ["D1", "D2", "D3"]
np.testing.assert_allclose(search_results["score"].to_numpy(), [0.8373573005465317, 0.319543268602613, 0.2561064480972134, 0.0, 0.0, 0.0])
print("All lecture-example checks passed.")

All lecture-example checks passed.


## Optional reference · scikit-learn's default convention
**Hidden lecture backup.** This section intentionally uses a different TF-IDF formula. Keep the outputs clearly labeled.

`TfidfTransformer()` uses count-based TF, `ln((1+N)/(1+df)) + 1` IDF, and L2 normalization. A term present in every document receives IDF **1**, so `campus` does not disappear. Setting only `smooth_idf=False` still leaves the added `+1`; it does not reproduce our classroom formula.

[TfidfTransformer documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html).

In [ ]:
default_transformer = TfidfTransformer()
tfidf_sklearn = default_transformer.fit_transform(X_counts)

default_idf = np.log((1 + N) / (1 + document_frequency)) + 1
np.testing.assert_allclose(default_transformer.idf_, default_idf)
manual_default = counts * default_idf
manual_default = manual_default / np.linalg.norm(manual_default, axis=1, keepdims=True)
np.testing.assert_allclose(tfidf_sklearn.toarray(), manual_default)

display(pd.DataFrame({
    "D1_classroom": tfidf_classroom[0],
    "D1_sklearn_defaults": tfidf_sklearn.toarray()[0]
}, index=vocabulary).round(3))
print("The default formula and normalization also match the library output.")

,D1_classroom,D1_sklearn_defaults
campus,0.000,0.158
library,0.136,0.586
quiet,0.222,0.714
study,0.043,0.247
space,0.043,0.247
cafe,0.000,0.000
coffee,0.000,0.000
food,0.000,0.000
gym,0.000,0.000
fitness,0.000,0.000


The default formula and normalization also match the library output.


## Interpretation questions
What evidence in the original descriptions explains D3's overlap with both library and cafe documents? What would a word-only representation miss about disagreement or negation? Which choices would need to change for a corpus in another language?

## Sources and provenance
The shared corpus, calculated tables, and prediction prompts were newly designed for this Week 4 revision. The conceptual sequence and the relative-TF/log10-IDF convention build on the Week 3 and Week 4 course decks.

Technical references: [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html), [TfidfTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html), [cosine similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html), and Manning, Raghavan & Schütze, *Introduction to Information Retrieval*, [TF-IDF weighting](https://nlp.stanford.edu/IR-book/html/htmledition/tf-idf-weighting-1.html). Jaccard's set-based definition is retained from the course's similarity deck, slides 47–51.